# 🧠 Data Friday Course: Cross-Dataset Validation & Integration

## Course Module 3: Advanced Multi-Dataset Analysis

Welcome to the **most advanced module** of our brain disease analysis course! Here we combine insights from multiple datasets to create robust, generalizable models.

## Learning Objectives
After completing this module, you will master:
- **Cross-dataset validation**: Ensuring models work across studies
- **Multi-task learning**: Training on multiple related problems
- **Advanced neural networks**: MLPs for complex pattern recognition
- **Feature overlap analysis**: Finding universal disease signatures
- **Permutation importance**: Understanding what models actually learn

## Why Cross-Dataset Analysis Matters

### The Replication Crisis in Science
- Many research findings don't replicate across different studies
- Single-dataset models often overfit to specific conditions
- Cross-validation ensures robust, generalizable results

### Clinical Translation Requirements
- **FDA approval**: Requires validation across multiple clinical sites
- **Population diversity**: Models must work across different demographics
- **Technical robustness**: Must handle different sequencing protocols

## What We'll Accomplish
1. **Load pre-processed datasets** from our previous analyses
2. **Identify shared genes** between cerebral and AD datasets
3. **Build cross-validated models** using advanced algorithms
4. **Discover universal biomarkers** that work across studies
5. **Validate with neural networks** for complex pattern recognition

## Advanced Techniques Covered
- **Support Vector Machines (SVM)** with L1 regularization
- **Multi-layer Perceptrons (MLP)** for non-linear patterns
- **Permutation importance** for model interpretability
- **Gene overlap analysis** for biological validation

## Resources for Advanced Learning
- [Cross-validation best practices](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Multi-task learning in ML](https://en.wikipedia.org/wiki/Multi-task_learning)
- [Neural networks for genomics](https://www.nature.com/articles/s41576-019-0122-6)

---

# Section 1: Loading Pre-Processed Datasets

## 📊 Integrating Multiple Studies

### The Power of Meta-Analysis
We're now combining results from our previous analyses:
- **Cerebral dataset**: General brain disease analysis
- **Alzheimer's dataset**: Disease-specific focus
- **Combined approach**: Leveraging strengths of both

### Loading Strategy
```python
# Load cerebral analysis results
temp_cereb_path = os.path.join(tempfile.gettempdir(), "cereb_combined.h5ad")
cereb_data = sc.read_h5ad(temp_cereb_path)

# Load Alzheimer's analysis results  
temp_combined_path = os.path.join(tempfile.gettempdir(), "adata_combined.h5ad")
adata = sc.read_h5ad(temp_combined_path)
```

### Why Use Pre-Processed Data?
1. **Quality control**: Already filtered and cleaned
2. **Feature selection**: Contains only disease-relevant genes
3. **Standardization**: Consistent preprocessing across datasets
4. **Efficiency**: Skip redundant computational steps

### Cross-Study Validation Benefits
- **Reduces overfitting**: Models must work on independent data
- **Increases confidence**: Consistent results across studies
- **Improves generalizability**: Works for new, unseen patients
- **Clinical relevance**: Closer to real-world deployment

**Best practice**: Always validate biomedical models on independent datasets!

In [1]:
import scanpy as sc
import tempfile
import requests
import os 


temp_combined_path = os.path.join(tempfile.gettempdir(), "adata_combined.h5ad")
adata= sc.read_h5ad(temp_combined_path)

temp_cereb_path = os.path.join(tempfile.gettempdir(), "cereb_combined.h5ad")
cereb_data = sc.read_h5ad(temp_cereb_path)

In [2]:
shared_genes = list(set(adata.var_names) & set(cereb_data.var_names))
adata_sub = adata[:, shared_genes]
cereb_sub = cereb_data[:, shared_genes]

cereb_sub

View of AnnData object with n_obs × n_vars = 82862 × 120
    obs: 'orig_cluster', 'orig_sub_cluster', 'broad_lineage', 'author_cell_type', 'dev_state', 'subtype', 'precisest_label', 'tissue_id', 'batch', 'size_factor', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'is_primary_data', 'author_stage', 'tissue_fragment', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_counts', 'n_genes'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_cells'
    uns: 'batch_condition', 'cell_type_colors', 'citation', 'default_embedding', 'disease_colors', 'neighbors', 'organism', 'organism_ontology_term_id', 'pca', 'schema_reference', 'schema_ve

# Section 2: Gene Overlap Analysis

## 🧬 Finding Universal Disease Signatures

### The Challenge of Cross-Dataset Integration
Different studies often measure different sets of genes, use different protocols, and focus on different aspects of disease. We need to find the **common ground**.

### Gene Intersection Strategy
```python
shared_genes = list(set(adata.var_names) & set(cereb_data.var_names))
```

This finds genes that are:
- **Present in both datasets**: Measured consistently
- **High quality**: Passed filtering in both analyses  
- **Disease-relevant**: Selected as important features

### Why Gene Overlap Matters
1. **Reproducibility**: Genes found in multiple studies are more likely to be real
2. **Clinical translation**: Consistent biomarkers work across populations
3. **Biological validation**: Multiple studies converging on same genes
4. **Reduced false positives**: Single-study artifacts are filtered out

### Expected Outcomes
- **Large overlap**: Suggests robust, universal disease signatures
- **Small overlap**: May indicate study-specific effects or different disease aspects
- **Quality over quantity**: Even small overlaps can be highly significant

### Subsetting to Shared Features
```python
adata_sub = adata[:, shared_genes]
cereb_sub = cereb_data[:, shared_genes]
```

This creates **matched datasets** with identical gene sets, enabling direct comparison and combined analysis.

**Statistical note**: Gene overlap analysis is a form of intersection-based feature selection!

In [3]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

X_ad = adata_sub.X
y_ad = LabelEncoder().fit_transform(adata_sub.obs["disease"])  # e.g., AD vs Control

X_cereb = cereb_sub.X
if "development_stage" in cereb_sub.obs.columns:
    y_cereb = LabelEncoder().fit_transform(cereb_sub.obs["development_stage"])
else:
    y_cereb = np.zeros(X_cereb.shape[0])

print(X_ad)

#print(y_ad)

[[ 1.87939384 -0.28566793  1.53113334 ... -0.94200827 -0.80771344
   1.68508067]
 [ 2.19018843 -0.28566793 -0.99458796 ... -0.94200827 -0.80771344
  -0.73657895]
 [-0.60050543 -0.28566793  1.6975118  ...  1.77032425 -0.80771344
  -0.73657895]
 ...
 [-0.60050543 -0.28566793  0.68479065 ...  0.46019066  0.57967642
  -0.73657895]
 [ 0.97217396 -0.28566793  0.91565929 ...  0.43880695  0.99766623
   0.79916656]
 [-0.60050543 -0.28566793 -0.99458796 ... -0.94200827  0.71846657
   0.79994722]]


# Section 3: Advanced Data Preparation

## 🔄 Preparing for Multi-Task Learning

### Label Encoding for Machine Learning
Machine learning algorithms require **numerical labels**, not text categories:

```python
from sklearn.preprocessing import LabelEncoder
y_ad = LabelEncoder().fit_transform(adata_sub.obs["disease"])
```

**What this does:**
- Converts text labels (e.g., "Alzheimer disease", "Normal") to numbers (0, 1, 2...)
- Maintains consistency across datasets
- Enables mathematical operations on labels

### Handling Different Label Types
Our datasets may have different types of outcome variables:
- **AD dataset**: Disease categories (Normal, AD, Other)
- **Cerebral dataset**: May include development stages or other classifications

### The Multi-Task Challenge
We're setting up for **multi-task learning** where we:
1. **Task 1**: Classify Alzheimer's vs. controls
2. **Task 2**: Classify cerebral disease conditions
3. **Joint analysis**: Find genes important for both tasks

### Feature Matrix Preparation
```python
X_ad = adata_sub.X      # Alzheimer's dataset features
X_cereb = cereb_sub.X   # Cerebral dataset features
```

Both now have:
- **Same gene set**: Identical feature space
- **Numerical labels**: Ready for ML algorithms
- **Consistent format**: Enable direct comparison

**Advanced concept**: This setup enables transfer learning between related disease classification tasks!

In [4]:
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectFromModel

# SVM for AD
svm_ad = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_ad.fit(X_ad, y_ad)
selected_ad = np.abs(svm_ad.coef_).sum(axis=0)

# SVM for Cerebellum
svm_cereb = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_cereb.fit(X_cereb, y_cereb)
selected_cereb = np.abs(svm_cereb.coef_).sum(axis=0)

# Top genes
top_n = 50
genes_array = np.array(shared_genes)
top_ad_genes = genes_array[np.argsort(selected_ad)[-top_n:]]
top_cereb_genes = genes_array[np.argsort(selected_cereb)[-top_n:]]
overlap = set(top_ad_genes) & set(top_cereb_genes)
print(f"Overlap ({len(overlap)}): {overlap}")


KeyboardInterrupt: 

# Section 4: Support Vector Machine Analysis

## ⚡ Advanced Feature Selection with SVM

### Why Support Vector Machines?
SVMs are particularly powerful for genomics data because they:
- **Handle high dimensions**: Work well with thousands of genes
- **Find sparse solutions**: Identify minimal gene sets
- **Robust to noise**: Less affected by irrelevant features
- **Linear interpretability**: Can understand which genes matter most

### L1 Regularization (Lasso) Explained
```python
LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
```

**Key parameters:**
- **`penalty='l1'`**: Forces sparsity (selects subset of genes)
- **`C=0.01`**: Strong regularization (prevents overfitting)
- **`dual=False`**: Required for L1 penalty with many features
- **`max_iter=5000`**: Sufficient iterations for convergence

### The Gene Selection Process
1. **Train separate SVMs** on each dataset
2. **Extract coefficients** indicating gene importance
3. **Rank genes** by absolute coefficient values
4. **Find overlapping** top genes between datasets

### Biological Interpretation
```python
selected_ad = np.abs(svm_ad.coef_).sum(axis=0)
top_ad_genes = genes_array[np.argsort(selected_ad)[-top_n:]]
```

This identifies genes that are:
- **Maximally discriminative** for disease classification
- **Consistently important** across different datasets
- **Potential biomarkers** for clinical use

**Learn more**: [SVM for genomics](https://academic.oup.com/bioinformatics/article/18/suppl_1/S96/225769)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Combine inputs
X_combined = np.vstack([X_ad, X_cereb])
y_ad_combined = np.concatenate([y_ad, [-1]*X_cereb.shape[0]])
y_cereb_combined = np.concatenate([[-1]*X_ad.shape[0], y_cereb])
# Split the data
X_ad_train, X_ad_test, y_ad_train, y_ad_test = train_test_split(X_ad, y_ad, test_size=0.2, random_state=42)
X_cereb_train, X_cereb_test, y_cereb_train, y_cereb_test = train_test_split(X_cereb, y_cereb, test_size=0.2, random_state=42)
# Train separate MLPs (simplified multitask)
mlp_ad = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)

mlp_ad.fit(X_ad_train, y_ad_train)
print("AD classification report:")
print(classification_report(y_ad_test, mlp_ad.predict(X_ad_test)))

mlp_cereb = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
mlp_cereb.fit(X_cereb_train, y_cereb_train)
print("Cerebellum classification report:")
print(classification_report(y_cereb_test, mlp_cereb.predict(X_cereb_test)))


AD classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6020
           1       1.00      1.00      1.00      8939
           2       1.00      1.00      1.00      9383

    accuracy                           1.00     24342
   macro avg       1.00      1.00      1.00     24342
weighted avg       1.00      1.00      1.00     24342

Cerebellum classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13445
           1       1.00      1.00      1.00     13197
           2       1.00      1.00      1.00      7849
           3       1.00      1.00      1.00     14200
           4       1.00      1.00      1.00     15580
           5       1.00      1.00      1.00      4594
           6       1.00      1.00      1.00      8033
           7       1.00      1.00      1.00      9359
           8       1.00      1.00      1.00      3181
           9      

# Section 5: Neural Network Analysis

## 🧠 Deep Learning for Complex Pattern Recognition

### Why Neural Networks for Genomics?
While linear models (logistic regression, SVM) work well for many problems, neural networks can capture:
- **Non-linear relationships**: Complex gene interactions
- **Hidden patterns**: Subtle disease signatures
- **Multi-level features**: Hierarchical biological organization

### Multi-Layer Perceptron (MLP) Architecture
```python
MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
```

**Architecture explained:**
- **Input layer**: All shared genes (features)
- **Hidden layer 1**: 128 neurons (pattern detection)
- **Hidden layer 2**: 64 neurons (pattern refinement)  
- **Output layer**: Disease classes (predictions)

### Training Strategy
We train **separate models** for each dataset to:
1. **Compare performance**: Neural networks vs. linear models
2. **Understand complexity**: Do we need non-linear models?
3. **Validate robustness**: Consistent results across architectures?

### Interpreting Neural Network Results
Unlike linear models, neural networks are "black boxes," but we can still gain insights:
- **Performance metrics**: How well do they classify?
- **Feature importance**: Which genes matter most? (via permutation)
- **Consistency**: Do both datasets show similar patterns?

### Expected Outcomes
- **High accuracy**: Neural networks often outperform linear models
- **Overfitting risk**: May memorize rather than generalize
- **Computational cost**: Slower training than linear models

**Deep learning in genomics**: [Nature Reviews Genetics](https://www.nature.com/articles/s41576-019-0122-6)

In [ ]:
from sklearn.inspection import permutation_importance

# Convert to dense if needed
X_ad_test_array = X_ad_test.toarray() if hasattr(X_ad_test, "toarray") else X_ad_test
X_cereb_test_array = X_cereb_test.toarray() if hasattr(X_cereb_test, "toarray") else X_cereb_test

# Permutation importance on test sets only
perm_ad = permutation_importance(mlp_ad, X_ad_test_array, y_ad_test, n_repeats=10, random_state=42)
top_nn_ad = genes_array[np.argsort(perm_ad.importances_mean)[-top_n:]]

perm_cereb = permutation_importance(mlp_cereb, X_cereb_test_array, y_cereb_test, n_repeats=10, random_state=42)
top_nn_cereb = genes_array[np.argsort(perm_cereb.importances_mean)[-top_n:]]

# Overlapping genes in NN-based importance
overlap_nn = set(top_nn_ad) & set(top_nn_cereb)
print(f"Overlap in NN-important genes: {overlap_nn}")


# Section 6: Model Interpretability & Course Conclusion

## 🔍 Understanding What Neural Networks Learn

### Permutation Importance: Opening the Black Box
Neural networks are powerful but opaque. **Permutation importance** helps us understand which features actually matter:

#### How It Works:
1. **Baseline performance**: Measure model accuracy on test data
2. **Shuffle one feature**: Randomly permute values for one gene
3. **Measure performance drop**: How much does accuracy decrease?
4. **Repeat for all features**: Get importance score for each gene
5. **Rank by importance**: Most important genes cause biggest performance drops

### Why Permutation Importance?
- **Model-agnostic**: Works with any algorithm
- **Real-world relevant**: Shows what models actually use
- **Robust**: Less affected by model artifacts
- **Interpretable**: Direct connection to prediction accuracy

### Cross-Method Validation
Comparing gene importance across methods:
- **SVM coefficients**: Linear feature weights
- **Neural network permutation**: Non-linear feature importance
- **Overlap analysis**: Genes important in multiple approaches

### What Overlap Means
```python
overlap_nn = set(top_nn_ad) & set(top_nn_cereb)
```

Genes in the overlap are:
- **Universally important**: Critical across datasets
- **Robust biomarkers**: Validated by multiple approaches
- **Therapeutic targets**: Prime candidates for drug development

## 🏆 Course Completion: What You've Mastered

### Technical Skills Gained:
1. ✅ **Single-cell RNA-seq analysis** with scanpy
2. ✅ **Machine learning** for biomedical data
3. ✅ **Cross-dataset validation** techniques
4. ✅ **Feature selection** and gene discovery
5. ✅ **Neural networks** for genomics
6. ✅ **Model interpretability** methods

### Biological Insights:
1. 🧬 **Disease signatures** at cellular resolution
2. 🎯 **Universal biomarkers** across studies
3. 🧠 **Cell-type specific** disease effects
4. 💊 **Therapeutic targets** for brain diseases

### Professional Applications:
- **Academic research**: Publish high-impact papers
- **Biotech industry**: Develop diagnostic tools
- **Pharmaceutical**: Discover drug targets
- **Clinical translation**: Improve patient care

**Congratulations!** You've completed a comprehensive course in computational neurobiology and biomedical data science!